In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
from src import operators as op


In [2]:

rng = np.random.default_rng(0)
pop = op.init_population(5, 3, (-5.12, 5.12), rng)
print(pop)
print(pop.shape)                                   # should be (5, 3)
print(pop.min() >= -5.12, pop.max() <= 5.12)       # should be True True


def sphere(x):
    return np.sum(x ** 2)

fitness = np.array([sphere(x) for x in pop])
print(fitness)

[[ 1.40248768 -2.35738405 -4.70043111]
 [-4.95075701  3.20788725  4.22661711]
 [ 1.09195034  2.35004478  0.44671991]
 [ 4.45514162  3.23434039 -5.09195776]
 [ 3.65981979 -4.77608371  2.35167177]]
(5, 3)
True True
[29.61828392 52.6648278   6.91462472 56.23727845 41.73561663]


In [3]:
parents = op.tournament_selection(pop, fitness, 5, 3, rng)
print(parents)
print([sphere(p) for p in parents])

[[ 1.40248768 -2.35738405 -4.70043111]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.40248768 -2.35738405 -4.70043111]]
[29.6182839174636, 6.9146247228092435, 6.9146247228092435, 6.9146247228092435, 29.6182839174636]


In [4]:
p1 = np.array([2.0, 0.0, -1.0])
p2 = np.array([4.0, 1.0, -1.0])
c1, c2 = op.blx_alpha_crossover(p1, p2, (-5.12, 5.12), rng)
print(c1, c2)

[ 3.58875805  0.73077022 -1.        ] [ 4.98883974  1.46167068 -1.        ]


In [5]:
c1, c2 = op.sbx_crossover(p1, p2, (-5.12, 5.12), rng)
print(c1, c2)
print((c1 + c2) / 2, (p1 + p2) / 2)   # should match

[ 1.97737357 -0.01500312 -1.        ] [ 4.02262643  1.01500312 -1.        ]
[ 3.   0.5 -1. ] [ 3.   0.5 -1. ]


In [6]:
kids = op.crossover(parents, 0.9, "sbx", (-5.12, 5.12), rng)
print(kids.shape)                                         # same as parents.shape
kids_none = op.crossover(parents, 0.0, "sbx", (-5.12, 5.12), rng)
print(np.array_equal(kids_none, parents))                 # rate 0 -> True
# op.crossover(parents, 0.9, "abc", (-5.12, 5.12), rng)     # should raise a clear error

(5, 3)
True


In [7]:
m = op.gaussian_mutation(pop, 0.3, (-5.12, 5.12), rng)
print((m != pop).mean())                  # roughly 0.3 (fraction of genes changed)
print(m.min() >= -5.12, m.max() <= 5.12)  # True True

0.3333333333333333
True True


In [8]:
m = op.polynomial_mutation(pop, 1.0, (-5.12, 5.12), rng)   # mutate every gene
print(np.round(m - pop, 2))       # changes: small, both + and -
print((m == 5.12).mean())         # should be ~0, NOT most genes stuck at the edge

[[ 0.93 -0.69  0.29]
 [ 0.9   1.26 -1.58]
 [ 0.61  1.48  1.13]
 [-0.57  1.32  0.71]
 [ 0.49 -0.02 -0.37]]
0.0


In [10]:
new = op.init_population(5, 3, (-5.12, 5.12), rng)
new_fit = np.array([sphere(x) for x in new])
p, f = op.apply_elitism(pop, fitness, new, new_fit, 2)
print(fitness.min(), f.min())    # f.min() can never be worse than fitness.min()
print(np.allclose(f, [sphere(x) for x in p]))   # True: scores match the solutions

6.9146247228092435 5.782564571403218
True


In [11]:
from src import ga
res = ga.run(sphere, (-5.12, 5.12), 10, {"seed": 0})
print(res)

{'best_f': 14.325347328513885, 'evals': 100, 'history': [14.325347328513885], 'time_s': 0.0005936000015935861}


In [12]:
res = ga.run(sphere, (-5.12, 5.12), 10, {"seed": 0, "max_evals": 20000})
print(res["best_f"], res["evals"])    # best_f should be close to 0, evals 20000
print(res["history"][:5], res["history"][-1])   # numbers going down

3.313731139030706e-06 20000
[14.325347328513885, 13.05121251183364, 11.523947875941627, 10.398803070651972, 9.507394022046281] 3.313731139030706e-06
